# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [1]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [2]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['SBAEEZTQGK', 'CJCSUSGNPT'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[19,  2,  1,  5,  5, 26, 20, 17,  7, 11],
       [ 3, 10,  3, 19, 21, 19,  7, 14, 16, 20]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0, 11,  7, 17, 20, 26,  5,  5,  1,  2],
       [ 0, 20, 16, 14,  7, 19, 21, 19,  3, 10]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[11,  7, 17, 20, 26,  5,  5,  1,  2, 19],
       [20, 16, 14,  7, 19, 21, 19,  3, 10,  3]])>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [3]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.hidden = 128
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64, 
                                                    batch_input_shape=[None, None])
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        # Attention用的Dense层(双线性attention)
        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
        
    @tf.function
    def call(self, enc_ids, dec_ids):
        '''
        带Attention的Seq2Seq模型
        Args:
            enc_ids: encoder输入 shape(batch, enc_len)
            dec_ids: decoder输入 shape(batch, dec_len)
        Returns:
            logits: shape(batch, dec_len, vocab_size)
        '''
        # 1. Encoder
        enc_emb = self.embed_layer(enc_ids)  # (batch, enc_len, 64)
        enc_out, enc_state = self.encoder(enc_emb)  # enc_out: (batch, enc_len, 128)
        
        # 2. Decoder
        dec_emb = self.embed_layer(dec_ids)  # (batch, dec_len, 64)
        dec_out, dec_state = self.decoder(dec_emb, initial_state=[enc_state])
        # dec_out: (batch, dec_len, 128)
        
        # 3. Attention机制 (双线性attention)
        # 计算attention score: dec_out @ W @ enc_out^T
        # dec_out: (batch, dec_len, 128)
        # enc_out: (batch, enc_len, 128)
        
        # 将decoder hidden state通过一个Dense层
        dec_transformed = self.dense_attn(dec_out)  # (batch, dec_len, 128)
        
        # 计算attention scores: dec_transformed @ enc_out^T
        # (batch, dec_len, 128) @ (batch, 128, enc_len) = (batch, dec_len, enc_len)
        attn_scores = tf.matmul(dec_transformed, enc_out, transpose_b=True)
        
        # 计算attention权重 (softmax)
        attn_weights = tf.nn.softmax(attn_scores, axis=-1)  # (batch, dec_len, enc_len)
        
        # 计算context vector (加权求和encoder outputs)
        # (batch, dec_len, enc_len) @ (batch, enc_len, 128) = (batch, dec_len, 128)
        context = tf.matmul(attn_weights, enc_out)
        
        # 4. 结合context和decoder output
        # 简单拼接
        combined = tf.concat([dec_out, context], axis=-1)  # (batch, dec_len, 256)
        
        # 5. 输出层
        logits = self.dense(combined)  # (batch, dec_len, 27)
        
        return logits
    
    
    @tf.function
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids)
        enc_out, enc_state = self.encoder(enc_emb)
        return enc_out, [enc_out[:, -1, :], enc_state]
    
    def get_next_token(self, x, state, enc_out):
        '''
        单步解码 (带attention)
        Args:
            x: 当前输入token shape(batch,)
            state: decoder状态 shape(batch, hidden)
            enc_out: encoder所有输出 shape(batch, enc_len, hidden)
        Returns:
            out: 预测的下一个token shape(batch,)
            state: 更新后的状态
        '''
        # 1. Decoder单步
        inp_emb = self.embed_layer(x)  # (batch, emb_sz)
        h, state = self.decoder_cell.call(inp_emb, state)  # h: (batch, hidden)
        
        # 2. Attention
        # 将decoder hidden state通过Dense层
        h_transformed = self.dense_attn(h)  # (batch, hidden)
        
        # 计算attention scores
        # (batch, 1, hidden) @ (batch, hidden, enc_len) = (batch, 1, enc_len)
        h_exp = tf.expand_dims(h_transformed, axis=1)  # (batch, 1, hidden)
        attn_scores = tf.matmul(h_exp, enc_out, transpose_b=True)  # (batch, 1, enc_len)
        
        # Softmax得到权重
        attn_weights = tf.nn.softmax(attn_scores, axis=-1)  # (batch, 1, enc_len)
        
        # 加权求和得到context
        context = tf.matmul(attn_weights, enc_out)  # (batch, 1, hidden)
        context = tf.squeeze(context, axis=1)  # (batch, hidden)
        
        # 3. 结合context和decoder hidden state
        combined = tf.concat([h, context], axis=-1)  # (batch, 2*hidden)
        
        # 4. 输出
        logits = self.dense(combined)  # (batch, vocab_size)
        out = tf.argmax(logits, axis=-1)  # (batch,)
        
        return out, state

# Loss函数以及训练逻辑

In [4]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(2000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [5]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

step 0 : loss 3.3042476
step 500 : loss 1.4814909
step 1000 : loss 0.32050878
step 1500 : loss 0.117661834


<tf.Tensor: shape=(), dtype=float32, numpy=0.046120346>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [6]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1], enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, True, False, False, True, True, True, True, True, False, False, True, True, True, True, False, True, True, True, True, True, True, True, True, False, True, True, False, True, True, False, True]
[('FWPVEROKVICSHOYBLXMM', 'MMXLBYOHSCIVKOREVPWF'), ('HZKHJENOLRKDGTIDWHWI', 'IWHWDITGDKRLONEJHKZH'), ('TDYDEADKVYNHMXPMBTBX', 'XBTBMPXMHNYVKDAEDYDL'), ('IQCOSHGWVNDIWXXZCCQA', 'AQCCZXXWIDNVWGHSOCQI'), ('CNTURBJJTTTVNPHPWRMW', 'WMRWPHPNVTTTJJBRUTNC'), ('HASTGDQHRRBAJLESQGQF', 'FQGQSELJABRRHQDCAOZH'), ('RFZGPHVWKNINFOZLYXUG', 'GUXYLZOFNINKWVHPGZFR'), ('AOTSJLUBYAUTJOJSCEMM', 'MMECSJOJTUAYBULJSTOA'), ('IBDKVIOAAMMOKFMSYJMS', 'NKWFSMFKOMMAAOIVKDBH'), ('FAZOAAIJHSMYUQIJUNHC', 'CHNUJIQUYMSHJIAAOZTF'), ('ICXAFCLTSJPXMABCCYAS', 'SAYCCBAMXPJSTLCFAXCI'), ('DBAITMDWQXFWFIUCUQZM', 'MZQUCUIFWFXQWDMTIABD'), ('LXZGLGZPCPRUAESDLUJS', 'SJULDSEAURPCPZGLGZXL'), ('WOZOKSKGWAFDKTSFGNYE', 'EYNGFSTKDFAWGKSKOZOW'), ('TARBUOJJXMDXKGTMTEFI', 'IFETMTGKXDMXJJOUBRAT'), ('RSJTJFAFRHIEJMTHQRHL', 'LHRQHTMJEIHRFAFJTJSR')